<a href="https://www.kaggle.com/code/rentoda/japanese-wiki-table?scriptVersionId=335631281" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:
!pip install romkan2
!pip install namedivider-python
!pip install python-Levenshtein
!pip install -q kagglehub


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os, time, requests, pandas as pd
from io import StringIO
import os, time, random, requests, pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

os.makedirs("data/parts", exist_ok=True)

URL = "https://qlever.cs.uni-freiburg.de/api/wikidata"
HEADERS = {
    "User-Agent": "MyBot/1.0 (contact: your_email@example.com)",
    "Accept": "text/tab-separated-values",
}

PREFIX = """PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX schema: <http://schema.org/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
"""

# 共通の絞り込み(日本国籍・人間・jawiki記事あり)
BASE = """?person wdt:P31 wd:Q5 ; wdt:P27 wd:Q17 .
  ?article schema:about ?person ; schema:inLanguage "ja" ;
           schema:isPartOf <https://ja.wikipedia.org/> ."""

# 単値寄りの列(ベース表)
BASE_QUERY = PREFIX + f"""
SELECT ?person ?article ?personLabel ?personLabelEn ?genderLabel
       ?birthDate ?deathDate ?description ?image ?height ?weight WHERE {{
  {BASE}
  OPTIONAL {{ ?person rdfs:label ?personLabel . FILTER(LANG(?personLabel)="ja") }}
  OPTIONAL {{ ?person rdfs:label ?personLabelEn . FILTER(LANG(?personLabelEn)="en") }}
  OPTIONAL {{ ?person wdt:P21 ?g . ?g rdfs:label ?genderLabel . FILTER(LANG(?genderLabel)="ja") }}
  OPTIONAL {{ ?person wdt:P569 ?birthDate }}
  OPTIONAL {{ ?person wdt:P570 ?deathDate }}
  OPTIONAL {{ ?person schema:description ?description . FILTER(LANG(?description)="ja") }}
  OPTIONAL {{ ?person wdt:P18 ?image }}
  OPTIONAL {{ ?person wdt:P2048 ?height }}
  OPTIONAL {{ ?person wdt:P2067 ?weight }}
}}"""

# 多値列: name -> (property, ラベルを取るか)
MULTI = {
    "readings": ("P1814", 0),
    "occupations": ("P106", 1),
    "causesOfDeath": ("P509", 1),
    "mannersOfDeath": ("P1196", 1),
    "birthPlaces": ("P19", 1),
    "deathPlaces": ("P20", 1),
    "almaMaters": ("P69", 1),
    "notableWorks": ("P800", 1),
    "genres": ("P136", 1),
    "parties": ("P102", 1),
    "spouses": ("P26", 1),
    "children": ("P40", 1),
    "fathers": ("P22", 1),
    "mothers": ("P25", 1),
    "siblings": ("P3373", 1),
    "awards": ("P166", 1),
    "positionsHeld": ("P39", 1),
    "militaryRanks": ("P410", 1),
    "militaryBranches": ("P241", 1),
}


def multi_query(pid, ent):
    body = (
        f'?person wdt:{pid} ?v . ?v rdfs:label ?l . FILTER(LANG(?l)="ja")'
        if ent
        else f"?person wdt:{pid} ?l ."
    )
    # GROUP BY なし: 生のペアを返し、集約は pandas 側で行う
    return PREFIX + f"SELECT ?person ?l WHERE {{ {BASE} {body} }}"


def classify(r):
    t = r.text
    if r.status_code == 200:
        return (
            "truncated"
            if "An error has occurred while exporting" in t[-2000:]
            else "success"
        )
    if "text/html" in r.headers.get("Content-Type", ""):
        return "rate limit"
    if "were busy" in t[:1000]:
        return "busy"
    if "Operation timed out" in t[:1000]:
        return "timeout"
    return "other: " + t[:200].replace("\n", " ")


MAX_WORKERS = 3   # 公開サーバーなので控えめに

def fetch(query, label, max_retries=12):
    for i in range(1, max_retries + 1):
        t0 = time.time()
        try:
            r = requests.post(URL, data={"query": query}, headers=HEADERS, timeout=300)
            reason = classify(r)
            print(f"[{label}] {i}: HTTP {r.status_code} | {reason} | "
                  f"{time.time()-t0:.1f}s | {len(r.text)/1e6:.1f} MB")
            if reason == "success":
                return r.text
            if r.status_code in (400, 403):
                raise RuntimeError(r.text[:500])
        except requests.exceptions.RequestException as e:
            print(f"[{label}] network error: {e}")
        # busy は即返るので短く待つ。長引くほど少しずつ伸ばす(上限30秒)
        wait = min(2 + i * 2, 30) + random.uniform(0, 2)
        time.sleep(wait)
    raise RuntimeError(f"{label}: all attempts failed")

def get_part(name, query):
    path = f"data/parts/{name}.tsv"
    if not os.path.exists(path):
        text = fetch(query, name)
        with open(path, "w", encoding="utf-8") as f:
            f.write(text)
    df = pd.read_csv(path, sep="\t", dtype=str)
    df.columns = [c.lstrip("?") for c in df.columns]
    return name, df

# --- 並列取得 ---
jobs = {"base": BASE_QUERY}
jobs.update({n: multi_query(pid, ent) for n, (pid, ent) in MULTI.items()})

parts = {}
t0 = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    # base を最初に投入(一番重いので先に走らせる)
    futures = [ex.submit(get_part, n, q) for n, q in jobs.items()]
    for f in as_completed(futures):
        name, d = f.result()
        parts[name] = d

# --- 結合 ---
df = parts["base"].drop_duplicates("person")
for name in MULTI:
    agg = (parts[name].dropna().drop_duplicates()
                      .groupby("person")["l"].agg("|".join).rename(name))
    df = df.merge(agg, on="person", how="left")

df.to_csv("data/jawikidata.csv", index=False, encoding="utf-8")
print(f"done: {len(df)} rows, {len(df.columns)} columns, {time.time()-t0:.0f}s")

In [17]:
import pandas as pd

wikidata_raw = pd.read_csv("data/jawikidata.csv", encoding="utf-8")

rename_map = {
    # Basic information
    "person": "qid_url",  # 237k
    "personLabel": "label",  # 236k
    "personLabelEn": "name_en",  # 226k
    "readings" : "readings",  # 193k
    "article": "wiki_url",  # 237k
    "genderLabel": "gender",  # 223k
    "birthDate": "birth_date",  # 200k
    "deathDate": "death_date",  # 65k
    "description": "description",  # 181k
    "image": "image_url",  # 35k
    # Career/Attributes
    "occupations": "occupations",  # 211k
    "birthPlaces": "birth_places",  # 137k
    "almaMaters": "alma_maters",  # 97k
    "positionsHeld": "positions_held",  # 16k
    "genres": "genres",  # 12k
    "deathPlaces": "death_places",  # 9.2k
    # physical
    "height": "height_cm",  # 28k
    "weight": "weight_kg",  # 16k
    "causesOfDeath": "death_causes",  # 5.8k
    "mannersOfDeath": "death_manners",  # 6.7k
    # family
    "fathers": "fathers",  # 8.9k
    "mothers": "mothers",  # 2.7k
    "spouses": "spouses",  # 6.8k
    "siblings": "siblings",  # 5.4k
    "children": "children",  # 7.1k
    # others
    "awards": "awards",  # 17k
    "parties": "parties",  # 4.8k
    "notableWorks": "notable_works",  # 3.5k
    "militaryBranches": "military_branches",  # 3.1k
    "militaryRanks": "military_ranks",  # 3.0k
}

wikidata_raw = wikidata_raw.rename(columns=rename_map)

In [ ]:

import romkan2
import unicodedata
import numpy as np
import re
import regex
import Levenshtein

def swap_name_order(name):
    if not isinstance(name, str):
        return name
    parts = name.split()
    if len(parts) == 2:
        return f"{parts[1]} {parts[0]}"
    elif len(parts) != 2:
        return ""
    return name


MACRON_MAP = {
    "ā": "aa", "ī": "ii", "ū": "uu", "ē": "ee", "ō": "ou",
    "Ā": "Aa", "Ī": "Ii", "Ū": "Uu", "Ē": "Ee", "Ō": "Ou",
}

# Passport-style long-vowel spellings, e.g. Ohno -> Ono. Can false-positive
# mid-word (e.g. "Shinohara"); mitigated by comparing against readings below.
_OH_LONG_VOWEL_RE = re.compile(r"(?i)oh(?=[a-z])")
_DZU_RE = re.compile(r"(?i)dzu")


def _expand_passport_style(name):
    name = _OH_LONG_VOWEL_RE.sub("ou", name)
    name = _DZU_RE.sub("zu", name)
    return name


def to_hiragana(name, passport_style: bool = True):
    if not isinstance(name, str):
        return name
    for macron, expanded in MACRON_MAP.items():
        name = name.replace(macron, expanded)
    nfkd_form = unicodedata.normalize("NFKD", name)
    name = "".join([c for c in nfkd_form if not unicodedata.combining(c)])
    # romkan2 reads a bare "-" as a long-vowel mark, corrupting hyphenated
    # names (e.g. "Takahito-shinnō" -> "たかひとーしんおう"); it's always
    # just a word-internal separator here, so drop it instead.
    name = name.replace("-", "")
    if passport_style:
        name = _expand_passport_style(name)
    return romkan2.to_hiragana(name)


def katakana_to_hiragana(text):
    if not isinstance(text, str):
        return text
    return "".join(
        chr(ord(c) - 0x60) if "ァ" <= c <= "ヶ" else c for c in text
    )


def has_latin_letter(text):
    if not isinstance(text, str):
        return False
    return bool(re.search(r"[a-zA-Zａ-ｚＡ-Ｚ]", text))


def has_kanji(text):
    if not isinstance(text, str):
        return False
    return bool(regex.search(r"\p{Script=Han}", text))


# Hiragana reading of a leading kabuki/rakugo-style generation prefix
# (初代/二代目/五代目/十二代目 etc), mirroring the kanji-side prefix strip in
# cell-8 -- but readings (P1814) aren't run through that step, so it survives
# here and inflates the word count (e.g. "ごだいめ ばんどう みのすけ").
_GENERATION_READING_RE = re.compile(
    r"^(?:しょだい|(?:いち|に|さん|よん|よ|ご|ろく|しち|なな|はち|きゅう|く|じゅう)+だいめ)\s+"
)


def strip_generation_reading(text):
    if not isinstance(text, str):
        return text
    return _GENERATION_READING_RE.sub("", text)


def _candidates(name_en):
    # swap x passport-style
    variants = []
    for order_fn in (swap_name_order, lambda x: x):
        for passport_style in (True, False):
            variants.append(to_hiragana(order_fn(name_en), passport_style))
    return variants


def _reinsert_space(alt, cand):
    # P1814 readings are sometimes one unbroken string with no separator;
    # borrow cand's split position when the flattened lengths match.
    if not isinstance(alt, str) or " " in alt or not isinstance(cand, str) or " " not in cand:
        return alt
    idx = cand.index(" ")
    cand_flat = cand.replace(" ", "")
    if len(cand_flat) == len(alt):
        return alt[:idx] + " " + alt[idx:]
    return alt


def best_hiragana(name_en, readings, threshold: float = 2.0):
    # Pick the candidate (of 4: swap x passport-style) closest on average to
    # all readings (P1814) alts; adopt the closest alt itself if within
    # threshold (empirically set via full-dataset distance-bucket audit).
    alts = []
    if isinstance(readings, str) and readings:
        alts = [
            strip_generation_reading(katakana_to_hiragana(a).replace("・", " "))
            for a in readings.split("|")
            if a
        ]

    if not isinstance(name_en, str):
        return alts[0] if alts else None

    cands = _candidates(name_en)

    if alts:
        alts_flat = [a.replace(" ", "") for a in alts]
        avg_dists = []
        per_cand_best_alt = []
        for c in cands:
            c_flat = c.replace(" ", "")
            per_alt_dists = [Levenshtein.distance(c_flat, a) for a in alts_flat]
            avg_dists.append(sum(per_alt_dists) / len(per_alt_dists))
            per_cand_best_alt.append(
                alts[min(range(len(alts)), key=lambda i: per_alt_dists[i])]
            )
        best_i = min(range(len(cands)), key=lambda i: avg_dists[i])
        # a leftover Latin letter means romanization conversion itself failed;
        # any real reading beats a broken candidate outright in that case
        if avg_dists[best_i] <= threshold or has_latin_letter(cands[best_i]):
            return _reinsert_space(per_cand_best_alt[best_i], cands[best_i])
        return cands[best_i]

    return cands[0]


def is_japanese_only(text):
    if not isinstance(text, str) or text == "":
        return True
    pattern = r"^[\p{Script=Han}\p{Script=Hiragana}\p{Script=Katakana}々ー\s]+$"
    return bool(regex.fullmatch(pattern, text))

def is_kanji_only(text):
    if not isinstance(text, str) or text == "":
        return True
    pattern = r"^[\p{Script=Han}々ー\s]+$"
    return bool(regex.fullmatch(pattern, text))

def has_english_or_num(text):
    if not isinstance(text, str) or text == "":
        return False
    pattern = r"[a-zA-Zａ-ｚＡ-Ｚ0-9０-９]"
    return bool(re.search(pattern, text))

def has_special_char(text):
    # symbols/punctuation/other-numerals(roman numerals etc)/combining marks/Greek/〆
    if not isinstance(text, str) or text == "":
        return False
    pattern = r"[\p{S}\p{P}\p{No}\p{M}\p{Script=Greek}〆]"
    return bool(regex.search(pattern, text))

In [ ]:
row_count = 0

def startL():
    global row_count
    row_count = wikidata["is_japanese_name"].sum()

def endL(task):
    global row_count
    current = wikidata["is_japanese_name"].sum()
    diff = row_count - current
    print(f'{diff} are not japanese name because "{task}"')
    row_count = current

In [ ]:
wikidata = wikidata_raw.copy()
wikidata["qid"] = wikidata["qid_url"].str.split("/").str[-1]

# missing label often means missing wikipedia page
wikidata = wikidata.dropna(subset=["label"], how="any")

wikidata["kanji"] = (
    wikidata["label"].str.replace(r"[\(（].*$", "", regex=True).str.strip()
)

# strip a leading kabuki/rakugo-style generation prefix, e.g. "三代目 市川市蔵" -> "市川市蔵"
wikidata["kanji"] = wikidata["kanji"].str.replace(
    r"^(?:[〇一二三四五六七八九十百千]+代目|初代|[0-9]+代目)\s*", "", regex=True
)

# "Family, Given" style commas break the 2-word split check below
wikidata["name_en"] = wikidata["name_en"].str.replace(",", "", regex=False)

wikidata["is_japanese_name"] = True
startL()

mask = wikidata["kanji"].apply(has_english_or_num)
wikidata.loc[mask, "is_japanese_name"] = False
wikidata.loc[mask, "kanji"] = None
endL("kanji with english or num")

mask = wikidata["kanji"].apply(lambda x: has_special_char(x) if pd.notna(x) else False)
wikidata.loc[mask, "is_japanese_name"] = False
wikidata.loc[mask, "kanji"] = None
endL("kanji with special char")

mask = wikidata["kanji"].apply(lambda x: not is_japanese_only(x) if pd.notna(x) else False)
wikidata.loc[mask, "is_japanese_name"] = False
wikidata.loc[mask, "kanji"] = None
endL("kanji not japanese")  # check none is dropped

# keep the old people with "no" in the reading by removing it
# (replace with a single space, not "", or the two remaining words merge into one)
mask = wikidata["name_en"].str.contains(" no ", case=True, na=False) & wikidata[
    "kanji"
].apply(lambda x: is_kanji_only(x) if pd.notna(x) else False)
wikidata.loc[mask, "name_en"] = wikidata.loc[mask, "name_en"].str.replace(
    " no ", " ", regex=False
)
endL("old people 'の'")  # check none is dropped

# strip a trailing roman-numeral generation suffix, e.g. "Nakamura Tomijuro IV"
mask = wikidata["name_en"].str.contains(r"\s[IVXLCDM]+$", regex=True, na=False) & wikidata[
    "kanji"
].apply(lambda x: is_kanji_only(x) if pd.notna(x) else False)
wikidata.loc[mask, "name_en"] = wikidata.loc[mask, "name_en"].str.replace(
    r"\s[IVXLCDM]+$", "", regex=True
)
endL("roman numeral generation suffix")

mask = ~((wikidata["name_en"].str.split().str.len() == 2) | wikidata["name_en"].isna())
wikidata.loc[mask, "is_japanese_name"] = False
wikidata.loc[mask, "name_en"] = None
endL("name_en which can't be split into 2")

wikidata["hiragana"] = wikidata.apply(
    lambda row: best_hiragana(row["name_en"], row["readings"]), axis=1
)

mask = wikidata["hiragana"].apply(lambda x: not is_japanese_only(x) if pd.notna(x) else False)
wikidata.loc[mask, "is_japanese_name"] = False
wikidata.loc[mask, "hiragana"] = None
endL("hiragana not japanese")

# hiragana must split into exactly 2 words (family + given), like kanji/name_en;
# catches garbled multi-part readings (e.g. historical titles with multiple "no")
mask = wikidata["hiragana"].apply(lambda x: len(x.split()) != 2 if pd.notna(x) else False)
wikidata.loc[mask, "is_japanese_name"] = False
wikidata.loc[mask, "hiragana"] = None
endL("hiragana which can't be split into 2")

In [ ]:
from functools import lru_cache
from namedivider import GBDTNameDivider

name_divider = GBDTNameDivider()


@lru_cache(maxsize=None)
def split_kanji(kanji):
    """Split a kanji-containing name into 'family given' via the ML divider.
    Leaves non-kanji (katakana/hiragana-only) names untouched -- namedivider
    is trained on kanji and splits those poorly."""
    if pd.isna(kanji) or len(kanji) <= 1 or not has_kanji(kanji):
        return kanji
    result = name_divider.divide_name(kanji)
    return f"{result.family} {result.given}"

In [ ]:
wikidata["kanji"] = wikidata["kanji"].apply(split_kanji)

In [ ]:
# kanji still has no Han character (namedivider skipped it above) -- fall back
# to the already-split hiragana (verified against name_en/readings) when it
# represents the same name, instead of guessing a split ourselves.
no_kanji = wikidata["kanji"].apply(lambda x: isinstance(x, str) and len(x) > 1 and not has_kanji(x))
kanji_as_hira = wikidata["kanji"].apply(katakana_to_hiragana)
hira_flat = wikidata["hiragana"].str.replace(" ", "", regex=False)
match = no_kanji & wikidata["hiragana"].str.contains(" ", na=False) & (kanji_as_hira == hira_flat)
wikidata.loc[match, "kanji"] = wikidata.loc[match, "hiragana"]

# kanji/hiragana must each contain a family-given space by this point,
# regardless of script; catches single-word pen names that had neither
# name_en nor readings to be checked against earlier (e.g. "ラムダ", "ポーキー").
mask = wikidata["kanji"].apply(lambda x: not isinstance(x, str) or " " not in x) | wikidata[
    "hiragana"
].apply(lambda x: not isinstance(x, str) or " " not in x)
wikidata.loc[mask, "is_japanese_name"] = False
wikidata.loc[mask, "kanji"] = None
wikidata.loc[mask, "hiragana"] = None
endL("kanji or hiragana without a space")

In [ ]:
# normalize gender to M/F; other minor categories are left untouched.
gender_map = {
    "男性": "M",
    "女性": "F",
}
wikidata["gender"] = wikidata["gender"].replace(gender_map)

In [ ]:
from datetime import datetime

wikidata["birth_year"] = pd.to_numeric(
    wikidata["birth_date"].str[:4], errors="coerce"
).astype("Int64")
wikidata["death_year"] = pd.to_numeric(
    wikidata["death_date"].str[:4], errors="coerce"
).astype("Int64")

wikidata["death_age"] = wikidata["death_year"] - wikidata["birth_year"]
wikidata["death_age"] = (
    wikidata["death_age"].where(wikidata["death_age"] > 0, pd.NA).astype("Int64")
)

current_year = datetime.now().year
cut_off_year = current_year - 100

wikidata["age"] = current_year - wikidata["birth_year"]

# dead
invalid = wikidata["death_age"].notna() & (wikidata["age"] > wikidata["death_age"])
wikidata["age"] = wikidata["age"].where(~invalid, wikidata["death_age"])

# age should be 0~120
wikidata["age"] = (
    wikidata["age"]
    .where((wikidata["age"] > 0) & (wikidata["age"] < 120), pd.NA)
    .astype("Int64")
)

In [ ]:
# 表1: death_age × age の na/notna クロス表
table1 = pd.crosstab(
    wikidata["age"].notna().map({True: "notna", False: "na"}),
    wikidata["death_age"].notna().map({True: "notna", False: "na"}),
    margins=True,
    margins_name="sum",
)
table1.index.name = "death_age"
table1.columns.name = "age"
table1

In [ ]:
# 表2: birth_year × death_year の na/notna クロス表
table2 = pd.crosstab(
    wikidata["birth_year"].notna().map({True: "notna", False: "na"}),
    wikidata["death_year"].notna().map({True: "notna", False: "na"}),
    margins=True,
    margins_name="sum",
)
table2.index.name = "birth_year"
table2.columns.name = "death_year"
table2

In [ ]:
# meters
wikidata["height_cm"] = np.where(
    wikidata["height_cm"] <= 3, wikidata["height_cm"] * 100, wikidata["height_cm"]
)

# feet
wikidata["height_cm"] = np.where(
    wikidata["height_cm"] <= 10, wikidata["height_cm"] * 30.48, wikidata["height_cm"]
)

In [ ]:
wikidata2 = wikidata.copy()
wikidata2["bmi"] = (wikidata2["weight_kg"] / (wikidata2["height_cm"] / 100) ** 2).round(1)

# original
player = wikidata2["occupations"].str.contains("選手", na=False)  # 29k
wikidata2[["weight_kg", "height_cm", "bmi"]][player].hist()
# height fix
height_mask = (wikidata2["height_cm"] < 90) & player
wikidata2.loc[height_mask, "height_cm"] = wikidata2.loc[height_mask, "height_cm"] * 2.54
wikidata2["bmi"] = (wikidata2["weight_kg"] / (wikidata2["height_cm"] / 100) ** 2).round(1)
wikidata2[["weight_kg", "height_cm", "bmi"]][player].hist()

# weight fix
weight_mask = (wikidata2["bmi"] >= 40) & player
wikidata2.loc[weight_mask, "weight_kg"] = (
    wikidata2.loc[weight_mask, "weight_kg"] * 0.453592
)

# checked all people with above 40 BMI and these are the people who was using lbs
# 渡辺華奈　丸藤正道　 沖田杏梨
mask = wikidata2["qid"].isin(["Q45819258", "Q1859112", "Q720246"])
wikidata2.loc[mask, "weight_kg"] = wikidata2.loc[mask, "weight_kg"] * 0.453592
wikidata2["bmi"] = (wikidata2["weight_kg"] / (wikidata2["height_cm"] / 100) ** 2).round(1)

wikidata2[["weight_kg", "height_cm", "bmi"]].hist()

In [ ]:
main_columns = [
    "qid",  # 236k
    "kanji",  # 236k
    "hiragana",  # 213k
    "description",  # 181k
    "gender",  # 223k
    "birth_year",  # 200k
    "death_year",  # 65k
    "age",  # 197k
    "death_age",  # 63k
    "label",  # 236k
    "name_en",  # 226k
    "is_japanese_name", # 236k
    "readings",  # 193k
    # Career/Attributes
    "occupations",  # 210k
    "birth_places",  # 137k
    "alma_maters",  # 97k
    "positions_held",  # 16k
    "genres",  # 12k
    "death_places",  # 9.2k
    # physical
    "height_cm",  # 28k
    "weight_kg",  # 16k
    "bmi",  # 16k
    "death_causes",  # 5.8k
    "death_manners",  # 6.7k
    # family
    "fathers",  # 8.9k
    "mothers",  # 2.7k
    "spouses",  # 6.8k
    "siblings",  # 5.4k
    "children",  # 7.1k
    # others
    "awards",  # 17k
    "parties",  # 4.8k
    "notable_works",  # 3.5k
    "military_branches",  # 3.1k
    "military_ranks",  # 3.0k
    # url
    "wiki_url",  # 236k
    "image_url",  # 35k
]

common_columns = [
    "qid",  # 236k
    "kanji",  # 232k
    "hiragana",  # 213k
    "gender",  # 223k
    "birth_year",  # 200k
    "death_year",  # 65k
    "age",  # 197k
    "death_age",  # 63k
    "description",  # 181k
    "occupations",  # 210k
    "birth_places",  # 137k
    "alma_maters",  # 97k
    "height_cm",  # 28k
    "weight_kg",  # 16k
    "awards",  # 17k
    "is_japanese_name", # 236k
]

In [ ]:
import pandas as pd
import os
# wikidata = pd.read_csv("data/jpersonwiki.csv", encoding="utf-8", low_memory=False)
os.makedirs("data", exist_ok=True)
wikidata = wikidata[main_columns]

# duplicate qid rows (from unaggregated SPARQL vars like height/weight/image
# fanning out the GROUP BY): keep the first row, but fill any of its nulls
# from later duplicate rows for the same qid.
wikidata = wikidata.groupby("qid", as_index=False).first()

wikidata = wikidata.loc[
    wikidata.notna().sum(axis=1).sort_values(ascending=False).index
]
wikidata = wikidata.reset_index(drop=True)
wikidata.to_csv("data/jpersonwiki.csv", index=False)

person_ctn = len(wikidata)

In [ ]:
import os
import sys
import tempfile
import zipfile
from pathlib import Path
from datetime import datetime
import kagglehub

IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
version_notes = f"auto update {datetime.now().strftime('%Y/%m/%d')} {person_ctn:,} persons"

if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    KAGGLE_USERNAME = user_secrets.get_secret("KAGGLE_USERNAME")
    KAGGLE_KEY = user_secrets.get_secret("KAGGLE_KEY")
    DATASET_NAME = user_secrets.get_secret("KAGGLE_DATASET_NAME")
else:
    from dotenv import dotenv_values

    env = dotenv_values(".env")
    KAGGLE_USERNAME = env.get("KAGGLE_USERNAME")
    KAGGLE_KEY = env.get("KAGGLE_KEY")
    DATASET_NAME = env.get("KAGGLE_DATASET_NAME")

if not (KAGGLE_USERNAME and KAGGLE_KEY and DATASET_NAME):
    print("Kaggle credentials not found. Skipping dataset upload.")
    sys.exit(0)

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

handle = f"{KAGGLE_USERNAME}/{DATASET_NAME}"

with tempfile.TemporaryDirectory() as tmp:
    zip_path = Path(tmp) / "dataset.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write("data/jpersonwiki.csv", "jpersonwiki.csv")

    kagglehub.dataset_upload(
        handle=handle,
        local_dataset_dir=str(zip_path.parent),
        version_notes=version_notes,
    )

print(f"Uploaded new version to {handle} with notes: {version_notes}")
